# Task 1: GPT from scratch on TinyStories

**How to run (only this one file is needed):**
1. In Colab: File → Upload notebook → choose this file.
2. Runtime → Change runtime type → choose a **GPU**.
3. Runtime → Run all.

Everything is saved in Google Drive under `MyDrive/DATA266_task1/`. If Colab disconnects, just Run all again: training resumes from the last checkpoint.
At the end, a `task1_results.zip` downloads. Unzip it into the repo root to add the results to GitHub.

## 1. GPU

In [ ]:
!nvidia-smi

## 2. Working folder in Google Drive

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

WORK = "/content/drive/MyDrive/DATA266_task1"  # same folder layout as the GitHub repo
os.makedirs(f"{WORK}/task1_llm/scripts", exist_ok=True)
os.makedirs(f"{WORK}/task1_llm/kruthika/src", exist_ok=True)
%cd "{WORK}"

## 3. Install packages
PyTorch, NumPy and Matplotlib are already on Colab.

In [ ]:
!pip install -q pyyaml datasets

## 4. Code
Each cell writes one file of the repo (identical to the GitHub version).

### Dataset download
`task1_llm/scripts/download_tinystories.py`

In [ ]:
%%writefile task1_llm/scripts/download_tinystories.py
"""Download the TinyStories dataset from Hugging Face into task1_llm/data/tinystories/.

It keeps Hugging Face's two official splits:
    train       ~2.1M stories -> we sample our 100K training stories from here
    validation  ~22K stories  -> we sample our 10K validation stories from here
Because they are separate splits, validation stories are never used for training.

Usage (from the repo root):
    python task1_llm/scripts/download_tinystories.py
"""
from pathlib import Path

from datasets import load_dataset

# Pinned dataset version, so every team member gets exactly the same data.
REVISION = "f54c09fd23315a6f9c86f9dc80f725de7d8f9c64"

# find task1_llm folder
current_file = Path(__file__).resolve()
scripts_folder = current_file.parent
task1_llm = scripts_folder.parent
# find data folder in task1_llm
data_folder = task1_llm / "data"
# find tinystories folder in task1_llm/data
dataset_folder = data_folder / "tinystories"


def download_tinystories():
    """Download and save the dataset if it is not there yet; return its folder."""
    # create data folder if it does not exist
    data_folder.mkdir(parents=True, exist_ok=True)

    # check if tinystories exist, if not: download the dataset from hf and save it to the folder.
    if dataset_folder.is_dir():
        print("Dataset already exists.")
    else:
        dataset = load_dataset("roneneldan/TinyStories", revision=REVISION)
        dataset.save_to_disk(dataset_folder)
        print("Download complete")
        for split, part in dataset.items():
            print(f"  {split}: {len(part):,} stories")
    return dataset_folder


if __name__ == "__main__":
    download_tinystories()

### Config: model size, training settings, epochs
`task1_llm/kruthika/src/config.yaml`

In [ ]:
%%writefile task1_llm/kruthika/src/config.yaml
# Full Task 1 training run (sized for a Colab GPU).
# All paths are relative to the repo root.

run_name: gpt_L6_H6_D384
seed: 1337
device: auto              # cuda on Colab

data:
  data_dir: task1_llm/data/tinystories   # saved by task1_llm/scripts/download_tinystories.py
  n_train: 100000         # stories from the HF train split (brief: 100K train)
  n_val: 10000            # stories from the HF validation split, never trained on (brief: 10K)
  block_size: 256         # context window in characters (~ 2-3 sentences)

model:
  n_layer: 6              # transformer blocks
  n_head: 6               # attention heads per block (head size = 384 / 6 = 64)
  n_embd: 384             # width of every hidden vector
  dropout: 0.1            # light regularisation; the data is large relative to the model

train:
  epochs: 10              # brief: minimum 10 epochs
  batch_size: 128         # 128 x 256 = 32,768 characters per step
  lr: 1.0e-3              # peak learning rate, reached after warm-up
  min_lr: 1.0e-4          # cosine decay ends here (10% of peak)
  warmup_steps: 1000      # linear warm-up from 0 (about 4% of training)
  weight_decay: 0.1       # AdamW, on weight matrices only
  betas: [0.9, 0.95]
  grad_clip: 1.0          # clip global gradient norm for stability
  precision: bf16         # mixed precision if the GPU supports bf16, otherwise fp32
  compile: true           # torch.compile on CUDA for speed
  log_every: 100          # steps between training-log lines
  eval_every: 500         # steps between train/val loss estimates
  eval_batches: 50        # batches per estimate
  ckpt_every: 2000        # steps between resume checkpoints

generate:
  prompts:
    - "Once upon a time"
    - "One day, a little girl named Lily"
    - "The dog was very sad because"
    - "Tom and his mom went to the park."
    - "In a big forest, there lived"
  samples_per_prompt: 4   # temperature samples per prompt (+1 greedy each)
  max_new_tokens: 500
  temperature: 0.8
  top_k: null

# Smoke test only (smoke_test.py): these values replace the ones above for a ~1 minute check.
smoke:
  data: {n_train: 2000, n_val: 200, block_size: 64}
  model: {n_layer: 2, n_head: 4, n_embd: 64, dropout: 0.0}
  train: {batch_size: 16, steps: 200, lr: 1.0e-3, log_every: 25, eval_batches: 10}
  generate: {prompt: "Once upon a time", max_new_tokens: 200, temperature: 1.0, n_samples: 3}

### Task 1.1: data preprocessing
`task1_llm/kruthika/src/data.py`

In [ ]:
%%writefile task1_llm/kruthika/src/data.py
"""Task 1.1 - character-level data preprocessing for TinyStories.

Pipeline: raw stories -> clean text -> train/val split -> char vocab -> integer ids
-> random fixed-length (x, y) windows where y is x shifted left by one character.
"""
import random

import torch

# Special character placed between stories so the model can learn where a story ends.
# Cleaning removes all non-ASCII characters, so this can never collide with real text.
EOS = "§"

# The raw data contains UTF-8 text that was wrongly decoded as Windows-1252
# ("mojibake"), e.g. a curly quote became "â€œ". Map these back to plain ASCII.
# Order matters: the bare "â€" (broken closing quote) must be replaced last.
MOJIBAKE = [
    ("â€œ", '"'), ("â€™", "'"), ("â€˜", "'"), ("â€¦", "..."),
    ("â€”", "-"), ("â€“", "-"), ("â€", '"'),
    ("“", '"'), ("”", '"'), ("‘", "'"), ("’", "'"), ("…", "..."), ("—", "-"), ("–", "-"),
    ("\xa0", " "), ("\t", " "),
]


def clean_text(text):
    """Repair mojibake, then keep only printable ASCII plus newlines."""
    for bad, good in MOJIBAKE:
        text = text.replace(bad, good)
    text = "".join(c for c in text if c == "\n" or 32 <= ord(c) < 127)
    return text.strip()


def _sample_clean(split, n, seed, exclude=frozenset()):
    """Randomly pick n cleaned, non-empty stories from a Hugging Face split,
    skipping any story whose text is in `exclude`."""
    order = list(range(len(split)))
    random.Random(seed).shuffle(order)  # fixed seed -> the same "random" sample every run
    picked, pos = [], 0
    while len(picked) < n:
        if pos >= len(order):
            raise ValueError(f"split has only {len(picked)} usable stories, need {n}")
        chunk = order[pos:pos + n - len(picked) + 1000]  # small buffer for skipped stories
        pos += len(chunk)
        for text in split.select(chunk)["text"]:
            text = clean_text(text or "")  # drop empty/malformed entries
            if text and text not in exclude and len(picked) < n:
                picked.append(text)
    return picked


def load_stories(path, n_train, n_val, seed):
    """Load the saved Hugging Face dataset folder and return (train_texts, val_texts).

    Validation comes from the official `validation` split and training from the
    `train` split, so the two never share a source. The train split also contains
    exact copies of some validation stories, so any training story identical to a
    chosen validation story is skipped - the 10K validation stories stay unseen.
    """
    from datasets import load_from_disk

    ds = load_from_disk(str(path))
    val = _sample_clean(ds["validation"], n_val, seed)
    train = _sample_clean(ds["train"], n_train, seed, exclude=set(val))
    return train, val


def build_vocab(texts):
    """Every distinct character in the training text, plus EOS, sorted for a stable order."""
    chars = sorted(set("".join(texts)) | {EOS})
    char_to_idx = {c: i for i, c in enumerate(chars)}
    idx_to_char = {i: c for c, i in char_to_idx.items()}
    return char_to_idx, idx_to_char


def encode(text, char_to_idx):
    """str -> list[int]. Characters not in the vocabulary are skipped."""
    return [char_to_idx[c] for c in text if c in char_to_idx]


def decode(ids, idx_to_char):
    """list[int] -> str."""
    return "".join(idx_to_char[i] for i in ids)


def encode_split(texts, char_to_idx):
    """Join all stories (each followed by EOS) into one long 1-D tensor of ids."""
    return torch.tensor(encode(EOS.join(texts) + EOS, char_to_idx), dtype=torch.long)


def get_batch(ids, block_size, batch_size, device, generator=None):
    """Sample `batch_size` random windows of `block_size` characters.

    x = ids[i : i+block_size], y = ids[i+1 : i+block_size+1], so y[t] is the
    character that comes right after x[t] - the next-character prediction target.
    """
    starts = torch.randint(len(ids) - block_size - 1, (batch_size, 1), generator=generator)
    offsets = (starts + torch.arange(block_size)).to(ids.device)  # (B, T) positions to read
    # One gather for the whole batch; if `ids` already lives on the GPU nothing is copied.
    return ids[offsets].to(device), ids[offsets + 1].to(device)

### Task 1.2: GPT model
`task1_llm/kruthika/src/model.py`

In [ ]:
%%writefile task1_llm/kruthika/src/model.py
"""Task 1.2 - GPT-style character language model, built from scratch.

Only basic layers are used (nn.Linear, nn.Embedding, nn.LayerNorm, nn.Dropout);
attention, masking and the transformer block are written by hand.

    ids (B,T) -> token emb + position emb -> [Block] x n_layer -> LayerNorm -> LM head -> logits (B,T,V)
    Block:  x = x + Attention(LayerNorm(x))
            x = x + FeedForward(LayerNorm(x))
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    """Multi-head self-attention where each position can only look at itself and earlier positions."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        # One linear layer produces queries, keys and values for all heads at once.
        self.qkv = nn.Linear(n_embd, 3 * n_embd)
        self.proj = nn.Linear(n_embd, n_embd)  # mixes the heads' outputs back together
        self.attn_drop = nn.Dropout(dropout)
        self.resid_drop = nn.Dropout(dropout)
        # Lower-triangular matrix: mask[i, j] = 1 if j <= i (position i may attend to j).
        # A buffer moves with .to(device) and is saved in the state_dict, but is not trained.
        mask = torch.tril(torch.ones(block_size, block_size, dtype=torch.bool))
        self.register_buffer("mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_dim): each head works on its own slice of C.
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Scaled dot-product attention: how well each query matches each key.
        # Dividing by sqrt(head_dim) keeps scores from growing with dimension,
        # which would make softmax saturate and gradients vanish.
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)  # (B, h, T, T)
        # Causal mask: future positions get -inf, so softmax gives them weight 0.
        scores = scores.masked_fill(~self.mask[:, :, :T, :T], float("-inf"))
        weights = self.attn_drop(F.softmax(scores, dim=-1))
        out = weights @ v  # weighted average of values: (B, h, T, head_dim)

        out = out.transpose(1, 2).contiguous().view(B, T, C)  # concatenate heads
        return self.resid_drop(self.proj(out))


class FeedForward(nn.Module):
    """Position-wise MLP: expand to 4x width, GELU non-linearity, project back."""

    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    """Transformer block (pre-LayerNorm): attention = tokens talk to each other,
    feed-forward = each token processes what it gathered. Residual connections
    (x + ...) give gradients a direct path through deep stacks."""

    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ffn = FeedForward(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class GPT(nn.Module):
    def __init__(self, cfg):
        """cfg keys: vocab_size, block_size, n_layer, n_head, n_embd, dropout."""
        super().__init__()
        self.block_size = cfg["block_size"]
        n_embd = cfg["n_embd"]
        dropout = cfg.get("dropout", 0.0)

        self.tok_emb = nn.Embedding(cfg["vocab_size"], n_embd)  # what each character is
        self.pos_emb = nn.Embedding(self.block_size, n_embd)    # where it sits (learned)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [Block(n_embd, cfg["n_head"], self.block_size, dropout) for _ in range(cfg["n_layer"])]
        )
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, cfg["vocab_size"])  # hidden state -> score per character

        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        # Small random weights (GPT-2 style) keep the initial predictions close to
        # uniform, so the starting loss is about ln(vocab_size).
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        if isinstance(m, nn.Linear) and m.bias is not None:
            nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.block_size, f"sequence length {T} > block_size {self.block_size}"
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))  # (B, T, C)
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))  # (B, T, vocab_size)

        loss = None
        if targets is not None:
            # Cross-entropy over every position: flatten (B, T) into B*T predictions.
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """Append one sampled character at a time.

        temperature=0 -> greedy (always the most likely char); <1 sharper, >1 more random.
        top_k -> only sample from the k most likely characters.
        """
        for _ in range(max_new_tokens):
            context = idx[:, -self.block_size:]  # model only sees the last block_size chars
            logits, _ = self(context)
            logits = logits[:, -1, :]  # prediction for the next character only
            if temperature == 0:
                next_id = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / temperature
                if top_k is not None:
                    kth = torch.topk(logits, min(top_k, logits.size(-1))).values[:, [-1]]
                    logits = logits.masked_fill(logits < kth, float("-inf"))
                next_id = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

### Metrics
`task1_llm/kruthika/src/metrics.py`

In [ ]:
%%writefile task1_llm/kruthika/src/metrics.py
"""Task 1 metric helpers (formula-level only; model-agnostic).

Agree on these definitions with your teammate so numbers are comparable across members.
"""
import math

import torch


def perplexity(ce_loss):
    """exp(mean cross-entropy in nats)."""
    return math.exp(ce_loss)


def bits_per_char(ce_loss):
    """Cross-entropy converted from nats to bits (one token = one character here)."""
    return ce_loss / math.log(2)


def top1_accuracy(logits, targets):
    """Fraction of positions where argmax prediction == next character. logits (B,T,V), targets (B,T)."""
    return (logits.argmax(dim=-1) == targets).float().mean().item()


def count_parameters(model, trainable_only=True):
    return sum(p.numel() for p in model.parameters() if p.requires_grad or not trainable_only)


def _units(text, unit):
    return text.split() if unit == "word" else list(text)


def _ngrams(tokens, n):
    return [tuple(tokens[i:i + n]) for i in range(len(tokens) - n + 1)]


def distinct_n(texts, n, unit="word"):
    """Unique n-grams / total n-grams, pooled over all generated samples (higher = more diverse)."""
    grams = [g for t in texts for g in _ngrams(_units(t, unit), n)]
    return len(set(grams)) / len(grams) if grams else 0.0


def repeated_ngram_rate(texts, n=4, unit="word"):
    """Share of n-grams that already appeared earlier in the same sample, averaged over samples
    (higher = more looping/repetition)."""
    rates = []
    for t in texts:
        grams = _ngrams(_units(t, unit), n)
        if grams:
            rates.append(1 - len(set(grams)) / len(grams))
    return sum(rates) / len(rates) if rates else 0.0


def grad_norm(model):
    """Global L2 norm of all parameter gradients (call after backward, before optimizer.step)."""
    norms = [p.grad.detach().norm(2) for p in model.parameters() if p.grad is not None]
    return torch.norm(torch.stack(norms), 2).item() if norms else 0.0

### Logging, manifests, device
`task1_llm/kruthika/src/utils.py`

In [ ]:
%%writefile task1_llm/kruthika/src/utils.py
"""Run plumbing: config loading, seeding, device selection, raw logs and manifests.

All paths are resolved relative to the repo root so nothing personal is hard-coded.
"""
import hashlib
import json
import logging
import platform
import random
import subprocess
import sys
from datetime import datetime
from importlib import metadata
from pathlib import Path

import numpy as np
import torch
import yaml

REPO_ROOT = Path(__file__).resolve().parents[3]
MEMBER = "kruthika"
TASK = "task1"
RAW_LOG_DIR = REPO_ROOT / "reproducibility" / "raw_logs"
MANIFEST_DIR = REPO_ROOT / "reproducibility" / "manifests"


def repo_path(p):
    """Resolve a config path (relative to repo root) to an absolute Path."""
    p = Path(p)
    return p if p.is_absolute() else REPO_ROOT / p


def rel(p):
    """Repo-relative string form of a path, for logs/manifests (no personal paths)."""
    try:
        return str(Path(p).resolve().relative_to(REPO_ROOT))
    except ValueError:
        return str(p)


def load_config(path):
    with open(repo_path(path)) as f:
        return yaml.safe_load(f)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def device_name(device):
    if device.type == "cuda":
        return torch.cuda.get_device_name(device)
    if device.type == "mps":
        return f"Apple MPS ({_mac_chip()})"
    return platform.processor() or platform.machine()


def _mac_chip():
    try:
        return subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
    except Exception:
        return platform.machine()


def sync(device):
    """Wait for queued GPU work so wall-clock timings are honest."""
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def reset_peak_memory(device):
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)


def peak_memory_mb(device):
    """Peak accelerator memory (CUDA: true peak; MPS: current driver allocation) + process peak RSS."""
    out = {}
    if device.type == "cuda":
        out["peak_gpu_mem_mb"] = torch.cuda.max_memory_allocated(device) / 2**20
    elif device.type == "mps":
        out["mps_driver_mem_mb"] = torch.mps.driver_allocated_memory() / 2**20
    try:
        import resource
        rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
        out["peak_rss_mb"] = rss / 2**20 if sys.platform == "darwin" else rss / 2**10
    except ImportError:  # Windows
        pass
    return out


def run_id(run_name):
    return f"{TASK}_{MEMBER}_{run_name}_{datetime.now().strftime('%Y%m%d-%H%M%S')}"


def get_logger(rid):
    """Logger that writes to stdout and an append-only raw log file. Never edit these files."""
    RAW_LOG_DIR.mkdir(parents=True, exist_ok=True)
    log_path = RAW_LOG_DIR / f"{rid}.log"
    logger = logging.getLogger(rid)
    logger.setLevel(logging.INFO)
    logger.propagate = False
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
    for h in (logging.StreamHandler(sys.stdout), logging.FileHandler(log_path)):
        h.setFormatter(fmt)
        logger.addHandler(h)
    return logger, log_path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def _git(*args):
    try:
        return subprocess.check_output(["git", *args], cwd=REPO_ROOT, text=True,
                                       stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None


def environment_info(device):
    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "torch": torch.__version__,
        "numpy": np.__version__,
        "device": str(device),
        "device_name": device_name(device),
        "git_commit": _git("rev-parse", "HEAD"),
        "git_dirty": bool(_git("status", "--porcelain")),
        "packages": sorted(f"{d.metadata['Name']}=={d.version}" for d in metadata.distributions()),
    }


def write_manifest(rid, cfg, device, log_path, checkpoints=(), metrics=None, extra=None):
    """Record exactly what produced a result: env, config, log, checkpoint hashes, metrics."""
    MANIFEST_DIR.mkdir(parents=True, exist_ok=True)
    manifest = {
        "run_id": rid,
        "created": datetime.now().isoformat(timespec="seconds"),
        "environment": environment_info(device),
        "config": cfg,
        "raw_log": rel(log_path),
        "checkpoints": [{"path": rel(c), "sha256": sha256(c)} for c in checkpoints],
        "metrics": metrics or {},
        **(extra or {}),
    }
    path = MANIFEST_DIR / f"{rid}.json"
    path.write_text(json.dumps(manifest, indent=2, default=str))
    return path

### Task 1.3: training, evaluation, generation
`task1_llm/kruthika/src/train.py`

In [ ]:
%%writefile task1_llm/kruthika/src/train.py
"""Task 1.3 - train the GPT on TinyStories, then evaluate and generate.

Usage (from the repo root):
    python task1_llm/kruthika/src/train.py            # settings from src/config.yaml
    python task1_llm/kruthika/src/train.py --resume   # continue after a disconnect
    python task1_llm/kruthika/src/train.py --epochs 12  # change the number of epochs

One "epoch" = as many randomly sampled training characters as the training split
contains (steps_per_epoch = train_chars // (batch_size * block_size)).

Writes (paths relative to task1_llm/kruthika/):
    checkpoints/<run>_best.pt      model weights with the lowest val loss (small; commit this)
    checkpoints/<run>_resume.pt    model + optimizer + progress, for --resume (large; git-ignored)
    outputs/<run>/                 history CSV, loss curves, samples, metrics JSON
    metrics_report.csv             every Task 1 metric in one file
plus a raw log and a manifest in reproducibility/.
"""
import argparse
import contextlib
import csv
import json
import math
import sys
import time
from pathlib import Path

import torch

SRC = Path(__file__).resolve().parent
sys.path.insert(0, str(SRC))
sys.path.insert(1, str(SRC.parents[1] / "scripts"))  # task1_llm/scripts, for the downloader

import data  # noqa: E402
import metrics  # noqa: E402
import utils  # noqa: E402
from download_tinystories import download_tinystories  # noqa: E402
from model import GPT  # noqa: E402

MEMBER_DIR = utils.REPO_ROOT / "task1_llm" / utils.MEMBER


def lr_at(step, tc, total_steps):
    """Linear warm-up from 0 to lr, then cosine decay down to min_lr."""
    if step < tc["warmup_steps"]:
        return tc["lr"] * (step + 1) / tc["warmup_steps"]
    progress = (step - tc["warmup_steps"]) / max(1, total_steps - tc["warmup_steps"])
    return tc["min_lr"] + 0.5 * (tc["lr"] - tc["min_lr"]) * (1 + math.cos(math.pi * min(1.0, progress)))


def make_optimizer(model, tc):
    """AdamW; weight decay only on matrices (Linear/Embedding weights), not on biases or LayerNorm."""
    decay = [p for p in model.parameters() if p.requires_grad and p.dim() >= 2]
    no_decay = [p for p in model.parameters() if p.requires_grad and p.dim() < 2]
    groups = [{"params": decay, "weight_decay": tc["weight_decay"]},
              {"params": no_decay, "weight_decay": 0.0}]
    fused = {"fused": True} if next(model.parameters()).is_cuda else {}
    return torch.optim.AdamW(groups, lr=tc["lr"], betas=tuple(tc["betas"]), **fused)


def autocast(device, precision):
    if device.type == "cuda" and precision == "bf16":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return contextlib.nullcontext()


@torch.no_grad()
def estimate_loss(model, ids, T, B, n_batches, device, precision, gen):
    """Mean loss and top-1 accuracy over n_batches random windows (quick check during training)."""
    model.eval()
    losses, accs = [], []
    for _ in range(n_batches):
        x, y = data.get_batch(ids, T, B, device, generator=gen)
        with autocast(device, precision):
            logits, loss = model(x, y)
        losses.append(loss.item())
        accs.append(metrics.top1_accuracy(logits, y))
    model.train()
    return sum(losses) / len(losses), sum(accs) / len(accs)


@torch.no_grad()
def full_eval(model, ids, T, B, device, precision, max_windows=None):
    """Loss and top-1 accuracy over consecutive non-overlapping windows covering the split."""
    model.eval()
    n = (len(ids) - 1) // T
    if max_windows:
        n = min(n, max_windows)
    xs = ids[: n * T].view(n, T)
    ys = ids[1: n * T + 1].view(n, T)
    loss_sum, correct = 0.0, 0
    for i in range(0, n, B):
        x, y = xs[i:i + B].to(device), ys[i:i + B].to(device)
        with autocast(device, precision):
            logits, loss = model(x, y)
        loss_sum += loss.item() * y.numel()
        correct += (logits.argmax(-1) == y).sum().item()
    model.train()
    return loss_sum / (n * T), correct / (n * T)


def save_plots(history, evals, out_dir):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    steps = [h["step"] for h in history]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(steps, [h["train_loss"] for h in history], alpha=0.35, label="train (per logged step)")
    ax.plot([e["step"] for e in evals], [e["train_loss"] for e in evals], "o-", label="train (eval batches)")
    ax.plot([e["step"] for e in evals], [e["val_loss"] for e in evals], "o-", label="validation")
    ax.set(xlabel="step", ylabel="cross-entropy loss (nats)", title="Training and validation loss")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_dir / "loss_curves.png", dpi=150)
    plt.close(fig)

    fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    a1.plot(steps, [h["lr"] for h in history])
    a1.set(ylabel="learning rate", title="LR schedule and gradient norm")
    a2.plot(steps, [h["grad_norm"] for h in history])
    a2.set(xlabel="step", ylabel="grad norm (pre-clip)")
    for a in (a1, a2):
        a.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(out_dir / "lr_gradnorm.png", dpi=150)
    plt.close(fig)


def generate_samples(model, gc, c2i, i2c, device, log):
    """Sample from each prompt with temperature sampling and greedy decoding; time generation."""
    model.eval()
    results = {"sampled": [], "greedy": []}
    new_tokens, secs = 0, 0.0
    for prompt in gc["prompts"]:
        idx = torch.tensor([data.encode(prompt, c2i)], dtype=torch.long, device=device)
        runs = [("sampled", gc["temperature"])] * gc["samples_per_prompt"] + [("greedy", 0)]
        for kind, temp in runs:
            utils.sync(device)
            t0 = time.perf_counter()
            out = model.generate(idx, gc["max_new_tokens"], temperature=temp, top_k=gc.get("top_k"))
            utils.sync(device)
            secs += time.perf_counter() - t0
            new_tokens += gc["max_new_tokens"]
            text = data.decode(out[0].tolist(), i2c)
            results[kind].append({"prompt": prompt, "text": text})
            log.info(f"[{kind}] {text!r}")
    model.train()
    return results, new_tokens / secs


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="task1_llm/kruthika/src/config.yaml")
    ap.add_argument("--resume", action="store_true", help="continue from checkpoints/<run>_resume.pt")
    ap.add_argument("--epochs", type=int, help="override train.epochs from the config")
    args = ap.parse_args()

    cfg = utils.load_config(args.config)
    if args.epochs:
        cfg["train"]["epochs"] = args.epochs  # recorded in the log and manifest
    dc, mc, tc, gc = cfg["data"], cfg["model"], cfg["train"], cfg["generate"]
    run = cfg["run_name"]
    T, B = dc["block_size"], tc["batch_size"]
    rid = utils.run_id(run)
    log, log_path = utils.get_logger(rid)
    utils.set_seed(cfg["seed"])
    device = utils.pick_device(cfg.get("device", "auto"))
    precision = tc.get("precision", "fp32") if device.type == "cuda" else "fp32"
    if precision == "bf16" and not torch.cuda.is_bf16_supported():
        precision = "fp32"  # older GPUs have no bf16 support
    if device.type == "cuda":
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
    env = utils.environment_info(device)
    log.info(f"run_id={rid} config={args.config} resume={args.resume}")
    log.info(f"python={env['python']} torch={env['torch']} device={device} ({env['device_name']}) "
             f"precision={precision} git={env['git_commit']} dirty={env['git_dirty']}")
    log.info(f"config: {json.dumps(cfg)}")

    out_dir = MEMBER_DIR / "outputs" / run
    ckpt_dir = MEMBER_DIR / "checkpoints"
    proc_dir = MEMBER_DIR / "data_processed"
    for d in (out_dir, ckpt_dir, proc_dir):
        d.mkdir(parents=True, exist_ok=True)
    best_path, resume_path = ckpt_dir / f"{run}_best.pt", ckpt_dir / f"{run}_resume.pt"

    # ---------------- data ----------------
    t0 = time.perf_counter()
    data_dir = utils.repo_path(dc["data_dir"])
    if not data_dir.is_dir():
        download_tinystories()
    train_texts, val_texts = data.load_stories(data_dir, dc["n_train"], dc["n_val"], cfg["seed"])
    c2i, i2c = data.build_vocab(train_texts)
    V = len(c2i)
    train_ids = data.encode_split(train_texts, c2i)
    val_ids = data.encode_split(val_texts, c2i)
    if device.type == "cuda":  # whole dataset fits easily in GPU memory -> no per-step copies
        train_ids, val_ids = train_ids.to(device), val_ids.to(device)
    (proc_dir / "vocab.json").write_text(json.dumps({"char_to_idx": c2i}, indent=1))
    (proc_dir / "split_stats.json").write_text(json.dumps({
        "data_dir": dc["data_dir"], "seed": cfg["seed"],
        "n_train_stories": len(train_texts), "n_val_stories": len(val_texts),
        "train_chars": len(train_ids), "val_chars": len(val_ids), "vocab_size": V,
    }, indent=1))
    log.info(f"data: {len(train_texts)} train / {len(val_texts)} val stories | "
             f"{len(train_ids):,} / {len(val_ids):,} chars | vocab={V} | {time.perf_counter() - t0:.1f}s")

    # ---------------- model ----------------
    model_cfg = {**mc, "vocab_size": V, "block_size": T}
    model = GPT(model_cfg).to(device)
    n_params = metrics.count_parameters(model)
    log.info(f"model: {model_cfg} | parameters={n_params:,}")
    opt = make_optimizer(model, tc)

    steps_per_epoch = len(train_ids) // (B * T)
    total_steps = tc["epochs"] * steps_per_epoch
    log.info(f"schedule: {tc['epochs']} epochs x {steps_per_epoch} steps = {total_steps} steps "
             f"({total_steps * B * T / 1e6:.0f}M tokens), warmup={tc['warmup_steps']}")

    gen = torch.Generator().manual_seed(cfg["seed"])
    eval_gen = torch.Generator().manual_seed(cfg["seed"] + 1)
    step, history, evals, best_val = 0, [], [], float("inf")
    nan_steps, spikes, train_secs = 0, 0, 0.0
    if args.resume and resume_path.exists():
        ck = torch.load(resume_path, map_location=device, weights_only=False)
        model.load_state_dict(ck["model_state"])
        opt.load_state_dict(ck["opt_state"])
        gen.set_state(ck["gen_state"].cpu())  # RNG state must stay on CPU
        step, history, evals, best_val = ck["step"], ck["history"], ck["evals"], ck["best_val"]
        nan_steps, spikes, train_secs = ck["nan_steps"], ck["spikes"], ck["train_secs"]
        log.info(f"resumed from {utils.rel(resume_path)} at step {step}")

    train_model = torch.compile(model) if (tc.get("compile") and device.type == "cuda") else model

    def save_resume():
        torch.save({"model_state": model.state_dict(), "opt_state": opt.state_dict(),
                    "gen_state": gen.get_state(), "step": step, "history": history, "evals": evals,
                    "best_val": best_val, "nan_steps": nan_steps, "spikes": spikes,
                    "train_secs": train_secs, "model_cfg": model_cfg, "config": cfg}, resume_path)

    # ---------------- training loop ----------------
    utils.reset_peak_memory(device)
    ema = None
    train_model.train()
    utils.sync(device)
    t_last, tok_last = time.perf_counter(), 0
    while step < total_steps:
        lr = lr_at(step, tc, total_steps)
        for g in opt.param_groups:
            g["lr"] = lr
        x, y = data.get_batch(train_ids, T, B, device, generator=gen)
        with autocast(device, precision):
            _, loss = train_model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(model.parameters(), tc["grad_clip"]).item()
        loss_val = loss.item()
        if not (math.isfinite(loss_val) and math.isfinite(gnorm)):
            nan_steps += 1  # skip the update rather than corrupt the weights
            log.warning(f"step {step}: non-finite loss/grad (loss={loss_val}, grad={gnorm}) - update skipped")
            opt.zero_grad(set_to_none=True)
        else:
            opt.step()
            if ema is not None and step > tc["warmup_steps"] and loss_val > 1.3 * ema:
                spikes += 1
                log.warning(f"step {step}: loss spike {loss_val:.4f} vs running mean {ema:.4f}")
            ema = loss_val if ema is None else 0.98 * ema + 0.02 * loss_val
        step += 1
        tok_last += B * T

        if step % tc["log_every"] == 0 or step == total_steps:
            utils.sync(device)
            dt = time.perf_counter() - t_last
            train_secs += dt
            tps = tok_last / dt
            history.append({"step": step, "epoch": step / steps_per_epoch, "lr": lr,
                            "train_loss": loss_val, "grad_norm": gnorm, "tokens_per_s": tps})
            log.info(f"step {step:6d}/{total_steps} | ep {step / steps_per_epoch:5.2f} | loss {loss_val:.4f} "
                     f"| lr {lr:.2e} | grad {gnorm:.3f} | {tps / 1e3:.0f}K tok/s")
            t_last, tok_last = time.perf_counter(), 0

        if step % tc["eval_every"] == 0 or step == total_steps:
            tr_loss, tr_acc = estimate_loss(train_model, train_ids, T, B, tc["eval_batches"], device, precision, eval_gen)
            va_loss, va_acc = estimate_loss(train_model, val_ids, T, B, tc["eval_batches"], device, precision, eval_gen)
            evals.append({"step": step, "epoch": step / steps_per_epoch, "train_loss": tr_loss,
                          "val_loss": va_loss, "train_acc": tr_acc, "val_acc": va_acc})
            log.info(f"EVAL step {step} | train {tr_loss:.4f} | val {va_loss:.4f} | "
                     f"gap {va_loss - tr_loss:+.4f} | val acc {va_acc:.3f}")
            if va_loss < best_val:
                best_val = va_loss
                torch.save({"model_state": model.state_dict(), "model_cfg": model_cfg,
                            "char_to_idx": c2i, "step": step, "val_loss": va_loss, "config": cfg}, best_path)
                log.info(f"new best val {va_loss:.4f} -> {utils.rel(best_path)}")
            t_last = time.perf_counter()  # don't count eval time as training time

        if step % tc["ckpt_every"] == 0 or step == total_steps:
            save_resume()
            t_last = time.perf_counter()

    # ---------------- final evaluation (best checkpoint) ----------------
    peak_mem = utils.peak_memory_mb(device)
    best = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(best["model_state"])
    log.info(f"final evaluation uses best checkpoint (step {best['step']})")
    val_loss, val_acc = full_eval(model, val_ids, T, B, device, precision)
    val_windows = (len(val_ids) - 1) // T
    train_loss, train_acc = full_eval(model, train_ids, T, B, device, precision, max_windows=val_windows)
    samples, gen_tps = generate_samples(model, gc, c2i, i2c, device, log)
    sampled = [s["text"] for s in samples["sampled"]]
    greedy = [s["text"] for s in samples["greedy"]]
    losses = [h["train_loss"] for h in history]

    report = {
        "run_name": run,
        "checkpoint": utils.rel(best_path),
        "best_step": best["step"],
        "train_cross_entropy": train_loss,
        "val_cross_entropy": val_loss,
        "val_perplexity": metrics.perplexity(val_loss),
        "val_bits_per_char": metrics.bits_per_char(val_loss),
        "train_perplexity": metrics.perplexity(train_loss),
        "generalization_gap": val_loss - train_loss,
        "train_top1_acc": train_acc,
        "val_top1_acc": val_acc,
        "distinct_1": metrics.distinct_n(sampled, 1),
        "distinct_2": metrics.distinct_n(sampled, 2),
        "distinct_3": metrics.distinct_n(sampled, 3),
        "repeated_4gram_rate": metrics.repeated_ngram_rate(sampled, 4),
        "greedy_repeated_4gram_rate": metrics.repeated_ngram_rate(greedy, 4),
        "grad_norm_mean": sum(h["grad_norm"] for h in history) / len(history),
        "grad_norm_max": max(h["grad_norm"] for h in history),
        "loss_spikes": spikes,
        "nan_inf_steps": nan_steps,
        "final_logged_train_loss": losses[-1],
        "parameters": n_params,
        "train_tokens_per_s": total_steps * B * T / train_secs,
        "gen_tokens_per_s": gen_tps,
        "total_train_time_s": train_secs,
        "total_train_time_min": train_secs / 60,
        "total_steps": total_steps,
        "epochs": tc["epochs"],
        "tokens_seen": total_steps * B * T,
        **peak_mem,
        "device": env["device_name"],
    }
    for k, v in report.items():
        log.info(f"METRIC {k} = {v}")

    # ---------------- write outputs ----------------
    with open(out_dir / "train_history.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(history[0]))
        w.writeheader()
        w.writerows(history)
    with open(out_dir / "eval_history.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(evals[0]))
        w.writeheader()
        w.writerows(evals)
    save_plots(history, evals, out_dir)
    (out_dir / "samples.json").write_text(json.dumps(samples, indent=2))
    with open(out_dir / "samples.txt", "w") as f:
        for kind in ("sampled", "greedy"):
            for s in samples[kind]:
                f.write(f"=== {kind} | prompt: {s['prompt']!r} ===\n{s['text']}\n\n")
    (out_dir / "metrics.json").write_text(json.dumps(report, indent=2))
    with open(MEMBER_DIR / "metrics_report.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["metric", "value"])
        w.writerows(report.items())

    manifest = utils.write_manifest(rid, cfg, device, log_path, checkpoints=[best_path], metrics=report,
                                    extra={"kind": "train", "outputs": utils.rel(out_dir)})
    log.info(f"outputs  -> {utils.rel(out_dir)}")
    log.info(f"metrics  -> {utils.rel(MEMBER_DIR / 'metrics_report.csv')}")
    log.info(f"raw log  -> {utils.rel(log_path)}")
    log.info(f"manifest -> {utils.rel(manifest)}")


if __name__ == "__main__":
    main()

## 5. Download TinyStories (skipped if already in Drive)

In [ ]:
!python task1_llm/scripts/download_tinystories.py

## 6. Train
Change `EPOCHS` here; all other settings are in `config.yaml` above.

In [ ]:
EPOCHS = 10  # brief: minimum 10

!python -u task1_llm/kruthika/src/train.py --epochs {EPOCHS} --resume

## 7. Results

In [ ]:
import pandas as pd
from IPython.display import Image, display
OUT = "task1_llm/kruthika/outputs/gpt_L6_H6_D384"
display(Image(f"{OUT}/loss_curves.png"))
display(Image(f"{OUT}/lr_gradnorm.png"))
pd.set_option("display.max_rows", 100)
pd.read_csv("task1_llm/kruthika/metrics_report.csv")

In [ ]:
print(open(f"{OUT}/samples.txt").read()[:6000])

## 8. Download results
Zips logs, manifests, metrics, plots, samples, vocab and the best checkpoint
(not the large resume checkpoint or the dataset). Unzip into the repo root: the folders line up.

In [ ]:
!rm -f task1_results.zip
!zip -qr task1_results.zip reproducibility task1_llm/kruthika -x "*_resume.pt" "*__pycache__*" "task1_llm/kruthika/src/*"
!ls -lh task1_results.zip
from google.colab import files
files.download("task1_results.zip")